# 視覺.py
**電腦視覺與深度學習 › Day14 OpenCV 電腦視覺**  ·  ⚠️ 錯誤示範(有 bug,對照修正版看)

這支程式打開攝影機,把每一格畫面轉灰階、找邊緣、找輪廓,再用紅線畫出來。它延續像素展示,只是把一張圖換成源源不絕的畫面。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/0/5)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> 📷 Colab 沒有攝影機、也開不了視窗:下面會**改用範例圖片**代替鏡頭,影像直接顯示在格子底下(只顯示前幾張)。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week1/DAY7"
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '視覺.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

In [ ]:
#@title 1. Colab 沒有攝影機和視窗:改用範例圖片、影像直接顯示在下面
import cv2, numpy as np, urllib.request, os
from google.colab.patches import cv2_imshow
def _pyl_sample():
    img = np.full((480, 640, 3), 255, np.uint8)
    cv2.circle(img, (130, 140), 70, (0, 0, 255), -1)
    cv2.circle(img, (520, 360), 55, (255, 0, 0), -1)
    cv2.rectangle(img, (260, 60), (400, 200), (0, 160, 0), -1)
    cv2.fillPoly(img, [np.array([[90, 420], [230, 420], [160, 300]])], (0, 200, 255))
    cv2.fillPoly(img, [np.array([[300, 300], [430, 280], [450, 420], [320, 440]])], (200, 0, 200))
    return img
class _PylCam:
    def __init__(self, *a, **k): self.n = 0; self.img = _pyl_sample()
    def isOpened(self): return True
    def read(self):
        self.n += 1
        return (True, self.img.copy()) if self.n <= 30 else (False, None)
    def set(self, *a): return True
    def get(self, *a): return 0
    def release(self): pass
_pyl_VC = cv2.VideoCapture
cv2.VideoCapture = lambda src=0, *a, **k: _PylCam() if isinstance(src, int) else _pyl_VC(src, *a, **k)
_pyl_shown = [0]
def _pyl_imshow(name, img):
    _pyl_shown[0] += 1
    if _pyl_shown[0] <= 6:
        print('▼', name); cv2_imshow(img)
cv2.imshow = _pyl_imshow
cv2.waitKey = lambda *a: ord('q') if _pyl_shown[0] >= 6 else -1
cv2.destroyAllWindows = lambda *a: None
cv2.namedWindow = lambda *a, **k: None
cv2.createTrackbar = lambda *a, **k: None
cv2.getTrackbarPos = lambda *a, **k: 0
cv2.setMouseCallback = lambda *a, **k: None
cv2.resizeWindow = cv2.moveWindow = cv2.setWindowProperty = cv2.setWindowTitle = lambda *a, **k: None
_pyl_imread = cv2.imread
def _pyl_imread2(path, *a, **k):
    img = _pyl_imread(path, *a, **k)
    if img is None and not os.path.exists(str(path)):
        print('(找不到', path, ':課程 repo 沒附這張圖,改用範例圖片)')
        img = _pyl_sample()
    return img
cv2.imread = _pyl_imread2
print('已改用範例圖片代替攝影機')

### 打開攝影機
VideoCapture 括號裡的 0 代表電腦預設的鏡頭,通常是筆電內建那支。接著用 isOpened 檢查有沒有打開成功,打不開就印錯誤訊息並結束。

In [ ]:
import cv2

# 1. 建立 VideoCapture 物件。參數 0 通常代表內建的預設鏡頭。
# 如果有接外接鏡頭，可以嘗試改為 1, 2 等。
cap = cv2.VideoCapture(0)

# 檢查鏡頭是否成功開啟
if not cap.isOpened():
    print("錯誤：無法開啟視訊鏡頭。")
    exit()

print("鏡頭已開啟！按下 'q' 鍵可結束程式。")

### 一格一格讀畫面
while True 是一個不會自己停的迴圈,每跑一圈就讀一格畫面。cap.read 會回傳兩個東西:ret 代表有沒有讀成功,frame 就是這一格影像。

### 灰階與 Canny 邊緣
先把彩色畫面轉成灰階,三個顏色變成一個亮度。再用 Canny 找出亮度突然變化的地方,也就是邊緣。100 和 200 是低、高兩個門檻,門檻越低,找到的邊越多。

### 找輪廓、設定面積範圍
findContours 把邊緣連成一圈一圈的輪廓。RETR_EXTERNAL 表示只要最外圈。下面兩個數字是面積的下限和上限,太小的雜點和太大的背景都不要。

### 篩選輪廓並畫出來
這個迴圈一個一個檢查輪廓,用 contourArea 算面積。面積在範圍內的才算數,印出面積,再用 drawContours 畫上去。顏色寫成 0、0、255,因為 OpenCV 的順序是藍綠紅,所以這是紅色。

### 用影像矩找中心點
moments 會算出這個形狀的統計量,用 m10 除以 m00、m01 除以 m00,就得到中心點座標。如果 m00 是 0 就不能除,所以先檢查。最後用 putText 在中心點寫上編號和面積。

### ⚠️ 注意:檢查放得太晚了
這裡才檢查 ret 有沒有讀成功,但前面早就拿 frame 去轉灰階了。如果鏡頭突然斷線,frame 會是 None,程式在 cvtColor 那行就先當掉,根本走不到這裡。比較好的寫法是 cap.read 之後馬上檢查,就像霍夫圓那支程式一樣。

### 顯示畫面、按 q 離開
imshow 把畫好輪廓的畫面秀出來。waitKey(1) 每格只等 1 毫秒,同時看你有沒有按 q。按了 q 就跳出迴圈。

### 收拾:放掉鏡頭
最後一定要 release 放掉鏡頭,再把視窗都關掉。沒有放掉的話,下一支程式可能就打不開攝影機了。

In [ ]:
while True:
    # 2. 逐影格（Frame）讀取影像
    # ret 是布林值，代表成功讀取與否；frame 是該影格的影像矩陣
    ret, frame = cap.read()
    灰色的圖片 = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)#彩圖轉灰階
    邊緣檢測 = cv2.Canny(灰色的圖片, 100, 200)
    img_copy = frame.copy()
    #opencv找輪廓的指令
    contours, hierarchy = cv2.findContours(邊緣檢測, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    min_area = 500   # 最小面積限制
    max_area = 50000 # 最大面積限制 

    count = 0
    for i, cnt in enumerate(contours):
        # 計算該輪廓的面積
        area = cv2.contourArea(cnt)
        # 判斷面積是否在設定的區間內
        if min_area <= area <= max_area:
            count += 1
            print(f"符合輪廓 {count}: 面積 = {area:.2f}")
            
            # 5. 繪製符合條件的輪廓
            # 參數: (要畫在哪張圖, 輪廓列表, -1代表畫出該index的單個輪廓, 顏色(B,G,R), 線條粗細)
            cv2.drawContours(img_copy, [cnt], -1, (0, 0, 255), 2)  # 用紅色(0,0,255)畫出輪廓
            
            # 【可選進階】計算輪廓的中心點並加上文字標籤
            M = cv2.moments(cnt)
            if M["m00"] != 0:
                cX = int(M["m10"] / M["m00"])
                cY = int(M["m01"] / M["m00"])
                # 在中心點標註這是第幾個符合的物體與其面積
                cv2.putText(img_copy, f"#{count} (A:{int(area)})", (cX - 20, cY), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
    
    
    # 如果讀取失敗，就跳出迴圈
    if not ret:
        print("錯誤：無法接收影像畫面。")
        break

    # 3. 顯示影像畫面，視窗名稱為 'Webcam'
    cv2.imshow('Webcam', img_copy)

    # 4. 等待 1 毫秒，並偵測是否按下了 'q' 鍵
    # 0xFF 是為了只取最後一個位元組，確保在不同平台上鍵盤碼正確
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 5. 釋放鏡頭資源並關閉所有 OpenCV 視窗
cap.release()
cv2.destroyAllWindows()

## 小結
你看到了即時影像處理的基本流程:讀畫面、灰階、Canny、找輪廓、畫出來。下一支霍夫圓檢測會在這個流程上加入「專門找圓」的演算法。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/0/5) 看逐句拆解、做練習。